# 🚨 Motor de Alertas Automáticas — XPharmaCare

Este notebook genera alertas de negocio a partir de los archivos de salida del Motor de Retención.  
Cada alerta tiene **tipo, prioridad, mensaje, métrica, umbral y acción recomendada**.  
El resultado se exporta a `alertas_generadas.csv`.

---


In [ ]:
import pandas as pd
from datetime import datetime

# ── Cargar datos ──────────────────────────────────────────────
churn = pd.read_csv('output_churn_scores.csv')
uplift = pd.read_csv('output_uplift_scores.csv')
motor = pd.read_csv('output_motor_ofertas.csv')
priorizacion = pd.read_csv('output_indice_priorizacion.csv')
segmentos = pd.read_csv('output_segmentos_clientes.csv')
interacciones = pd.read_csv('fact_interacciones_callcenter.csv')
adherencia = pd.read_csv('fact_adherencia_terapeutica.csv')
transacciones = pd.read_csv('fact_transacciones.csv')
clientes = pd.read_csv('dataset_analitico_clientes.csv')

print(f"✅ Datos cargados — {len(clientes):,} clientes")


## ⚙️ Configuración de Umbrales

Aquí defines los parámetros que disparan cada tipo de alerta.  
Ajusta estos valores según las prioridades de negocio de XPharmaCare.


In [ ]:
# ══════════════════════════════════════════════════════════════
# UMBRALES CONFIGURABLES — Ajusta según tu negocio
# ══════════════════════════════════════════════════════════════

UMBRALES = {
    # ── Churn ─────────────────────────────────────────────────
    'churn_critico_prob': 0.80,          # Probabilidad para considerar riesgo CRÍTICO
    'churn_alto_prob': 0.60,             # Probabilidad para considerar riesgo ALTO
    'churn_critico_min_clientes': 100,   # Mínimo de clientes para disparar alerta masiva

    # ── Valor del cliente ─────────────────────────────────────
    'clv_alto_valor_min': 3000,          # CLV 24m mínimo para considerar "alto valor"
    'prioridad_alta_min': 50,            # Índice de priorización mínimo para alta prioridad

    # ── Uplift / Retención ────────────────────────────────────
    'pct_persuadibles_min': 0.20,        # Si % persuadibles < esto, la estrategia está fallando
    'ofertas_pendientes_max': 200,       # Máximo aceptable de ofertas sin ejecutar

    # ── Call Center / Sentimiento ─────────────────────────────
    'tasa_resolucion_meta': 0.85,        # Meta de resolución del call center
    'ratio_negatividad_max': 0.35,       # Ratio máximo aceptable de interacciones negativas
    'puntaje_neg_alerta': 0.50,          # Puntaje negativo promedio que dispara alerta

    # ── Adherencia Terapéutica ────────────────────────────────
    'adherencia_critica': 0.60,          # Por debajo de esto = adherencia crítica
    'adherencia_baja': 0.75,             # Por debajo de esto = adherencia baja
    'ratio_retraso_max': 0.40,           # Máximo aceptable de ciclos con retraso

    # ── Operativo ─────────────────────────────────────────────
    'dias_recencia_inactivo': 180,       # Días sin compra para considerar inactivo
    'costo_retencion_max_dop': 150000,   # Presupuesto máximo mensual de retención
}

print("⚙️ Umbrales configurados:")
for k, v in UMBRALES.items():
    print(f"   {k}: {v}")


## 🔧 Motor de Generación de Alertas

Cada función evalúa una condición específica y genera alertas con estructura uniforme:
- **tipo**: Categoría (CHURN, VALOR, UPLIFT, SENTIMIENTO, ADHERENCIA, OPERATIVO)
- **prioridad**: CRÍTICA, ALTA, MEDIA, BAJA
- **mensaje**: Descripción legible de la alerta
- **metrica_actual**: El valor que disparó la alerta
- **umbral**: El umbral configurado
- **clientes_afectados**: Cantidad de clientes impactados
- **accion_recomendada**: Qué hacer
- **impacto_estimado_dop**: Estimación del impacto financiero


In [ ]:
# ══════════════════════════════════════════════════════════════
# MOTOR DE ALERTAS
# ══════════════════════════════════════════════════════════════

alertas = []

def agregar_alerta(tipo, prioridad, mensaje, metrica_actual, umbral,
                   clientes_afectados=0, accion='', impacto_dop=0):
    """Registra una alerta en la lista global."""
    alertas.append({
        'fecha_generacion': datetime.now().strftime('%Y-%m-%d %H:%M'),
        'tipo': tipo,
        'prioridad': prioridad,
        'mensaje': mensaje,
        'metrica_actual': round(metrica_actual, 4) if isinstance(metrica_actual, float) else metrica_actual,
        'umbral': umbral,
        'clientes_afectados': clientes_afectados,
        'accion_recomendada': accion,
        'impacto_estimado_dop': round(impacto_dop, 2),
    })

print("🔧 Función de alertas lista")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. ALERTAS DE CHURN
# ══════════════════════════════════════════════════════════════

# 1a. Volumen de clientes en riesgo CRÍTICO
criticos = churn[churn['probabilidad_churn'] >= UMBRALES['churn_critico_prob']]
n_criticos = len(criticos)

if n_criticos >= UMBRALES['churn_critico_min_clientes']:
    # Estimar ingreso en riesgo cruzando con priorización
    ids_criticos = criticos['customer_id'].values
    clv_en_riesgo = priorizacion[priorizacion['customer_id'].isin(ids_criticos)]['clv_24m'].sum()

    agregar_alerta(
        tipo='CHURN',
        prioridad='CRÍTICA',
        mensaje=f'{n_criticos:,} clientes con probabilidad de fuga ≥ {UMBRALES["churn_critico_prob"]*100:.0f}%',
        metrica_actual=n_criticos,
        umbral=UMBRALES['churn_critico_min_clientes'],
        clientes_afectados=n_criticos,
        accion='Activar protocolo de retención inmediata — priorizar persuadibles de alto valor',
        impacto_dop=clv_en_riesgo,
    )

# 1b. Clientes de ALTO VALOR en riesgo crítico (los más peligrosos)
alto_valor_ids = priorizacion[priorizacion['clv_24m'] >= UMBRALES['clv_alto_valor_min']]['customer_id']
av_en_riesgo = churn[
    (churn['customer_id'].isin(alto_valor_ids)) &
    (churn['probabilidad_churn'] >= UMBRALES['churn_critico_prob'])
]
n_av_riesgo = len(av_en_riesgo)

if n_av_riesgo > 0:
    clv_av = priorizacion[priorizacion['customer_id'].isin(av_en_riesgo['customer_id'])]['clv_24m'].sum()
    agregar_alerta(
        tipo='CHURN',
        prioridad='CRÍTICA',
        mensaje=f'{n_av_riesgo} clientes de ALTO VALOR (CLV ≥ RD${UMBRALES["clv_alto_valor_min"]:,}) en riesgo crítico de fuga',
        metrica_actual=n_av_riesgo,
        umbral=0,
        clientes_afectados=n_av_riesgo,
        accion='Escalar a gerencia — ejecutar Descuento+Llamada VIP inmediatamente',
        impacto_dop=clv_av,
    )

# 1c. Alerta de riesgo ALTO (nivel intermedio)
altos = churn[
    (churn['probabilidad_churn'] >= UMBRALES['churn_alto_prob']) &
    (churn['probabilidad_churn'] < UMBRALES['churn_critico_prob'])
]
n_altos = len(altos)

if n_altos >= UMBRALES['churn_critico_min_clientes']:
    agregar_alerta(
        tipo='CHURN',
        prioridad='ALTA',
        mensaje=f'{n_altos:,} clientes con riesgo alto de fuga ({UMBRALES["churn_alto_prob"]*100:.0f}%-{UMBRALES["churn_critico_prob"]*100:.0f}%)',
        metrica_actual=n_altos,
        umbral=UMBRALES['churn_critico_min_clientes'],
        clientes_afectados=n_altos,
        accion='Activar campañas preventivas — emails de reactivación y recordatorios',
        impacto_dop=priorizacion[priorizacion['customer_id'].isin(altos['customer_id'])]['clv_24m'].sum(),
    )

# 1d. Tasa general de churn
tasa_churn = clientes['churn'].mean()
if tasa_churn > 0.40:
    agregar_alerta(
        tipo='CHURN',
        prioridad='ALTA',
        mensaje=f'Tasa de churn general en {tasa_churn*100:.1f}% — supera el 40% tolerable',
        metrica_actual=tasa_churn,
        umbral=0.40,
        clientes_afectados=int(clientes['churn'].sum()),
        accion='Revisar estrategia global de retención — posible problema sistémico',
        impacto_dop=0,
    )

print(f"✅ Churn: {len(alertas)} alertas generadas hasta ahora")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. ALERTAS DE UPLIFT / RETENCIÓN
# ══════════════════════════════════════════════════════════════

# 2a. Proporción de persuadibles
n_total = len(uplift)
n_persuadibles = len(uplift[uplift['tipo_respuesta'] == 'Persuadible'])
pct_persuadibles = n_persuadibles / n_total

agregar_alerta(
    tipo='UPLIFT',
    prioridad='MEDIA' if pct_persuadibles >= UMBRALES['pct_persuadibles_min'] else 'ALTA',
    mensaje=f'{n_persuadibles:,} clientes persuadibles ({pct_persuadibles*100:.1f}%) — solo estos responden a ofertas',
    metrica_actual=pct_persuadibles,
    umbral=UMBRALES['pct_persuadibles_min'],
    clientes_afectados=n_persuadibles,
    accion='Concentrar 100% del presupuesto de retención en este segmento',
    impacto_dop=0,
)

# 2b. Sleeping Dogs — clientes que EMPEORAN con la oferta
n_sleeping = len(uplift[uplift['tipo_respuesta'] == 'Sleeping Dog'])
pct_sleeping = n_sleeping / n_total

if pct_sleeping > 0.15:
    agregar_alerta(
        tipo='UPLIFT',
        prioridad='ALTA',
        mensaje=f'{n_sleeping:,} Sleeping Dogs ({pct_sleeping*100:.1f}%) — contactarlos DESTRUYE valor',
        metrica_actual=pct_sleeping,
        umbral=0.15,
        clientes_afectados=n_sleeping,
        accion='Excluir de TODAS las campañas de retención — agregar a lista de no contactar',
        impacto_dop=0,
    )

# 2c. Ofertas pendientes de ejecución
ofertas_pendientes = motor[motor['accion_recomendada'].isin([
    'Recordatorio+Descuento', 'Puntos bonus+Oferta', 'Descuento+Llamada VIP', 'Envío gratis+Promo'
])]
n_pendientes = len(ofertas_pendientes)

if n_pendientes > UMBRALES['ofertas_pendientes_max']:
    costo_pendiente = ofertas_pendientes['costo_estimado_dop'].sum()
    agregar_alerta(
        tipo='UPLIFT',
        prioridad='ALTA',
        mensaje=f'{n_pendientes} ofertas de alto impacto pendientes de ejecución',
        metrica_actual=n_pendientes,
        umbral=UMBRALES['ofertas_pendientes_max'],
        clientes_afectados=n_pendientes,
        accion='Ejecutar ofertas priorizando clientes persuadibles con mayor CLV',
        impacto_dop=costo_pendiente,
    )

# 2d. Costo total de retención vs presupuesto
costo_total = motor['costo_estimado_dop'].sum()
if costo_total > UMBRALES['costo_retencion_max_dop']:
    agregar_alerta(
        tipo='OPERATIVO',
        prioridad='MEDIA',
        mensaje=f'Costo de retención RD${costo_total:,.0f} excede presupuesto de RD${UMBRALES["costo_retencion_max_dop"]:,}',
        metrica_actual=costo_total,
        umbral=UMBRALES['costo_retencion_max_dop'],
        clientes_afectados=0,
        accion='Recortar ofertas en Lost Causes y Sure Things — reasignar a persuadibles',
        impacto_dop=costo_total - UMBRALES['costo_retencion_max_dop'],
    )

print(f"✅ Uplift/Retención: {len(alertas)} alertas acumuladas")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. ALERTAS DE SENTIMIENTO / CALL CENTER
# ══════════════════════════════════════════════════════════════

# 3a. Tasa de resolución
tasa_resolucion = interacciones['resuelto'].mean()

if tasa_resolucion < UMBRALES['tasa_resolucion_meta']:
    brecha = UMBRALES['tasa_resolucion_meta'] - tasa_resolucion
    agregar_alerta(
        tipo='SENTIMIENTO',
        prioridad='ALTA' if brecha > 0.10 else 'MEDIA',
        mensaje=f'Tasa de resolución Call Center en {tasa_resolucion*100:.1f}% (meta: {UMBRALES["tasa_resolucion_meta"]*100:.0f}%)',
        metrica_actual=tasa_resolucion,
        umbral=UMBRALES['tasa_resolucion_meta'],
        clientes_afectados=int(len(interacciones) * (1 - tasa_resolucion)),
        accion='Revisar protocolos de atención — capacitar agentes en resolución de primer contacto',
        impacto_dop=0,
    )

# 3b. Ratio de negatividad global
n_negativas = len(interacciones[interacciones['sentimiento'] == 'NEGATIVO'])
ratio_neg = n_negativas / len(interacciones)

if ratio_neg > UMBRALES['ratio_negatividad_max']:
    agregar_alerta(
        tipo='SENTIMIENTO',
        prioridad='ALTA',
        mensaje=f'{n_negativas:,} interacciones negativas ({ratio_neg*100:.1f}%) — supera umbral de {UMBRALES["ratio_negatividad_max"]*100:.0f}%',
        metrica_actual=ratio_neg,
        umbral=UMBRALES['ratio_negatividad_max'],
        clientes_afectados=interacciones[interacciones['sentimiento']=='NEGATIVO']['customer_id'].nunique(),
        accion='Analizar motivos principales de fricción — escalar a operaciones',
        impacto_dop=0,
    )

# 3c. Motivos de fricción más frecuentes
motivos = interacciones[interacciones['sentimiento'] == 'NEGATIVO']['motivo'].value_counts()
top_motivos = motivos.head(3)

for motivo, count in top_motivos.items():
    pct_del_total = count / len(interacciones)
    if pct_del_total > 0.05:  # Si un solo motivo es >5% de todas las interacciones
        agregar_alerta(
            tipo='SENTIMIENTO',
            prioridad='MEDIA',
            mensaje=f'Motivo de fricción recurrente: "{motivo}" — {count} casos ({pct_del_total*100:.1f}% del total)',
            metrica_actual=count,
            umbral=int(len(interacciones) * 0.05),
            clientes_afectados=interacciones[
                (interacciones['motivo'] == motivo) & (interacciones['sentimiento'] == 'NEGATIVO')
            ]['customer_id'].nunique(),
            accion=f'Investigar causa raíz de "{motivo}" — crear plan correctivo específico',
            impacto_dop=0,
        )

# 3d. Puntaje negativo promedio alto
if 'puntaje_negativo' in interacciones.columns:
    neg_interactions = interacciones[interacciones['sentimiento'] == 'NEGATIVO']
    if len(neg_interactions) > 0:
        avg_neg = neg_interactions['puntaje_negativo'].mean()
        if avg_neg > UMBRALES['puntaje_neg_alerta']:
            agregar_alerta(
                tipo='SENTIMIENTO',
                prioridad='MEDIA',
                mensaje=f'Intensidad de negatividad promedio en {avg_neg:.2f} (umbral: {UMBRALES["puntaje_neg_alerta"]})',
                metrica_actual=avg_neg,
                umbral=UMBRALES['puntaje_neg_alerta'],
                clientes_afectados=len(neg_interactions),
                accion='Las quejas no solo son frecuentes sino intensas — revisar calidad de servicio',
                impacto_dop=0,
            )

print(f"✅ Sentimiento: {len(alertas)} alertas acumuladas")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. ALERTAS DE ADHERENCIA TERAPÉUTICA
# ══════════════════════════════════════════════════════════════

# 4a. Adherencia global
adh_global = adherencia['tasa_adherencia_periodo'].mean()

if adh_global < UMBRALES['adherencia_baja']:
    agregar_alerta(
        tipo='ADHERENCIA',
        prioridad='ALTA' if adh_global < UMBRALES['adherencia_critica'] else 'MEDIA',
        mensaje=f'Adherencia terapéutica global en {adh_global*100:.1f}% — por debajo de {UMBRALES["adherencia_baja"]*100:.0f}%',
        metrica_actual=adh_global,
        umbral=UMBRALES['adherencia_baja'],
        clientes_afectados=adherencia['customer_id'].nunique(),
        accion='Reforzar programa de recordatorios de receta — involucrar farmacéuticos',
        impacto_dop=0,
    )

# 4b. Adherencia por condición crónica
for condicion, grupo in adherencia.groupby('condicion_cronica'):
    adh_cond = grupo['tasa_adherencia_periodo'].mean()
    n_pacientes = grupo['customer_id'].nunique()

    if adh_cond < UMBRALES['adherencia_critica']:
        agregar_alerta(
            tipo='ADHERENCIA',
            prioridad='CRÍTICA',
            mensaje=f'Adherencia CRÍTICA en {condicion}: {adh_cond*100:.1f}% — {n_pacientes} pacientes',
            metrica_actual=adh_cond,
            umbral=UMBRALES['adherencia_critica'],
            clientes_afectados=n_pacientes,
            accion=f'Intervención urgente para pacientes de {condicion} — contacto farmacéutico directo',
            impacto_dop=0,
        )
    elif adh_cond < UMBRALES['adherencia_baja']:
        agregar_alerta(
            tipo='ADHERENCIA',
            prioridad='MEDIA',
            mensaje=f'Adherencia baja en {condicion}: {adh_cond*100:.1f}% — {n_pacientes} pacientes',
            metrica_actual=adh_cond,
            umbral=UMBRALES['adherencia_baja'],
            clientes_afectados=n_pacientes,
            accion=f'Activar recordatorios automáticos para pacientes de {condicion}',
            impacto_dop=0,
        )

# 4c. Pacientes crónicos con ratio de retraso alto
cronicos = clientes[clientes['es_cronico'] == True].copy()
if 'ratio_retraso' in cronicos.columns:
    retraso_alto = cronicos[cronicos['ratio_retraso'] > UMBRALES['ratio_retraso_max']]
    n_retrasados = len(retraso_alto)

    if n_retrasados > 0:
        agregar_alerta(
            tipo='ADHERENCIA',
            prioridad='ALTA',
            mensaje=f'{n_retrasados} pacientes crónicos con ratio de retraso > {UMBRALES["ratio_retraso_max"]*100:.0f}%',
            metrica_actual=n_retrasados,
            umbral=0,
            clientes_afectados=n_retrasados,
            accion='Contactar para reactivar tratamiento — posible abandono terapéutico',
            impacto_dop=priorizacion[priorizacion['customer_id'].isin(retraso_alto['customer_id'])]['clv_24m'].sum(),
        )

print(f"✅ Adherencia: {len(alertas)} alertas acumuladas")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. ALERTAS OPERATIVAS
# ══════════════════════════════════════════════════════════════

# 5a. Clientes inactivos (sin compra en X días)
inactivos = clientes[clientes['recencia_dias'] > UMBRALES['dias_recencia_inactivo']]
n_inactivos = len(inactivos)

if n_inactivos > 0:
    pct_inactivos = n_inactivos / len(clientes)
    agregar_alerta(
        tipo='OPERATIVO',
        prioridad='ALTA' if pct_inactivos > 0.25 else 'MEDIA',
        mensaje=f'{n_inactivos:,} clientes inactivos (>{UMBRALES["dias_recencia_inactivo"]} días sin compra) — {pct_inactivos*100:.1f}% de la cartera',
        metrica_actual=n_inactivos,
        umbral=UMBRALES['dias_recencia_inactivo'],
        clientes_afectados=n_inactivos,
        accion='Campaña de reactivación segmentada — email + oferta para persuadibles inactivos',
        impacto_dop=priorizacion[priorizacion['customer_id'].isin(inactivos['customer_id'])]['clv_24m'].sum(),
    )

# 5b. Clientes crónicos SIN programa de lealtad
cronicos_sin_lealtad = clientes[(clientes['es_cronico'] == True) & (clientes['programa_lealtad'] == False)]
n_csl = len(cronicos_sin_lealtad)

if n_csl > 0:
    agregar_alerta(
        tipo='OPERATIVO',
        prioridad='MEDIA',
        mensaje=f'{n_csl} pacientes crónicos NO están en programa de lealtad — oportunidad de retención',
        metrica_actual=n_csl,
        umbral=0,
        clientes_afectados=n_csl,
        accion='Enrolar automáticamente en programa de lealtad nivel Bronce — activar beneficios de crónico',
        impacto_dop=priorizacion[priorizacion['customer_id'].isin(cronicos_sin_lealtad['customer_id'])]['clv_24m'].sum(),
    )

# 5c. Concentración de ingreso (riesgo de dependencia)
top_pct = 0.05
n_top = max(1, int(len(clientes) * top_pct))
top_clientes = clientes.nlargest(n_top, 'total_gastado_dop')
ingreso_top = top_clientes['total_gastado_dop'].sum()
ingreso_total = clientes['total_gastado_dop'].sum()
concentracion = ingreso_top / ingreso_total if ingreso_total > 0 else 0

if concentracion > 0.40:
    agregar_alerta(
        tipo='OPERATIVO',
        prioridad='MEDIA',
        mensaje=f'Top {top_pct*100:.0f}% de clientes concentra {concentracion*100:.1f}% del ingreso — alto riesgo de dependencia',
        metrica_actual=concentracion,
        umbral=0.40,
        clientes_afectados=n_top,
        accion='Diversificar base de ingresos — desarrollar segmentos medios con campañas de frecuencia',
        impacto_dop=ingreso_top,
    )

print(f"✅ Operativo: {len(alertas)} alertas acumuladas")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 📋 CONSOLIDAR Y EXPORTAR
# ══════════════════════════════════════════════════════════════

df_alertas = pd.DataFrame(alertas)

# Ordenar por prioridad (CRÍTICA primero)
orden_prioridad = {'CRÍTICA': 0, 'ALTA': 1, 'MEDIA': 2, 'BAJA': 3}
df_alertas['_orden'] = df_alertas['prioridad'].map(orden_prioridad)
df_alertas = df_alertas.sort_values(['_orden', 'tipo']).drop(columns='_orden').reset_index(drop=True)

# Exportar
df_alertas.to_csv('alertas_generadas.csv', index=False, encoding='utf-8-sig')

print(f"🚨 Total: {len(df_alertas)} alertas generadas")
print(f"📁 Exportado a: alertas_generadas.csv")
print()
print("Distribución por prioridad:")
print(df_alertas['prioridad'].value_counts().to_string())
print()
print("Distribución por tipo:")
print(df_alertas['tipo'].value_counts().to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════
# 👁️ VISTA PREVIA DE ALERTAS
# ══════════════════════════════════════════════════════════════

# Mostrar alertas CRÍTICAS y ALTAS
alertas_urgentes = df_alertas[df_alertas['prioridad'].isin(['CRÍTICA', 'ALTA'])]

print(f"🔴 {len(alertas_urgentes)} alertas urgentes (CRÍTICA + ALTA):")
print("=" * 80)

for _, row in alertas_urgentes.iterrows():
    emoji = '🔴' if row['prioridad'] == 'CRÍTICA' else '🟠'
    print(f"\n{emoji} [{row['tipo']}] {row['prioridad']}")
    print(f"   {row['mensaje']}")
    print(f"   → Acción: {row['accion_recomendada']}")
    if row['clientes_afectados'] > 0:
        print(f"   → Clientes afectados: {row['clientes_afectados']:,}")
    if row['impacto_estimado_dop'] > 0:
        print(f"   → Impacto estimado: RD${row['impacto_estimado_dop']:,.0f}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 📊 RESUMEN EJECUTIVO
# ══════════════════════════════════════════════════════════════

impacto_total = df_alertas['impacto_estimado_dop'].sum()
n_criticas = len(df_alertas[df_alertas['prioridad'] == 'CRÍTICA'])
n_altas = len(df_alertas[df_alertas['prioridad'] == 'ALTA'])
clientes_unicos = df_alertas['clientes_afectados'].sum()

print("┌─────────────────────────────────────────────────────┐")
print("│           RESUMEN EJECUTIVO DE ALERTAS              │")
print("├─────────────────────────────────────────────────────┤")
print(f"│  Total alertas generadas:    {len(df_alertas):>6}                │")
print(f"│  Alertas CRÍTICAS:           {n_criticas:>6}                │")
print(f"│  Alertas ALTAS:              {n_altas:>6}                │")
print(f"│  Impacto financiero total:   RD${impacto_total:>12,.0f}     │")
print("├─────────────────────────────────────────────────────┤")
print("│  📁 Archivo: alertas_generadas.csv                  │")
print("└─────────────────────────────────────────────────────┘")
